License

- Content (explanatory text, figures): CC BY 4.0 — see /LICENSE-CONTENT
- Code cells and standalone code files: MIT License — see /LICENSE-CODE

Attribution example: Getman, R. (2026). CBE 3610 Python Notebooks and Course Materials. https://github.com/dr-rachel-bg-teaching/CBE3610


In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp

## Given data

In [ ]:
t=np.array([0,0.5,1,2,3,4,5,6,7,8,9,10,12,14])
C=np.array([0,0.6,1.4,5,8,10,8,6,4,3,2.2,1.6,0.6,0])
pd.DataFrame({"t (min)":t,"C (g/m^3)":C})

,t (min),C (g/m^3)
0,0.0,0.0
1,0.5,0.6
2,1.0,1.4
3,2.0,5.0
4,3.0,8.0
5,4.0,10.0
6,5.0,8.0
7,6.0,6.0
8,7.0,4.0
9,8.0,3.0


## $E(t)$

$$ E(t) = \frac{C(t)}{\int_0^\infty C(t) dt} $$

We already have $C(t)$. Use numerical integration to integrate it:

In [ ]:
intC = cumulative_trapezoid(C, t, initial=0)
E = C/intC[-1]
print("E =",E)

E = [0.         0.01173021 0.02737048 0.09775171 0.15640274 0.19550342
 0.15640274 0.11730205 0.07820137 0.05865103 0.04301075 0.03128055
 0.01173021 0.        ]


In [ ]:
pd.DataFrame({"t (min)":t,"C (g/m^3)":C,"E (unitless)":E})

,t (min),C (g/m^3),E (unitless)
0,0.0,0.0,0.000000
1,0.5,0.6,0.011730
2,1.0,1.4,0.027370
3,2.0,5.0,0.097752
4,3.0,8.0,0.156403
5,4.0,10.0,0.195503
6,5.0,8.0,0.156403
7,6.0,6.0,0.117302
8,7.0,4.0,0.078201
9,8.0,3.0,0.058651


## $F(t)$

$$ F(t) = \int_0^t E(t) dt $$

And we plot it as a function of t. First, take the integral of $E(t)$:

In [ ]:
F = cumulative_trapezoid(E, t, initial=0)
print("F(t) =",F)

F(t) = [0.         0.00293255 0.01270772 0.07526882 0.20234604 0.37829912
 0.5542522  0.69110459 0.7888563  0.8572825  0.90811339 0.94525904
 0.98826979 1.        ]


In [ ]:
pd.DataFrame({"t (min)":t,"C (g/m^3)":C,"E (unitless)":E,"F (unitless)":F})

,t (min),C (g/m^3),E (unitless),F (unitless)
0,0.0,0.0,0.000000,0.000000
1,0.5,0.6,0.011730,0.002933
2,1.0,1.4,0.027370,0.012708
3,2.0,5.0,0.097752,0.075269
4,3.0,8.0,0.156403,0.202346
5,4.0,10.0,0.195503,0.378299
6,5.0,8.0,0.156403,0.554252
7,6.0,6.0,0.117302,0.691105
8,7.0,4.0,0.078201,0.788856
9,8.0,3.0,0.058651,0.857283


# Computing X

The reaction is 2A $\longrightarrow$ B. It exhibits elementary kinetics with $k$ = 0.1 m$^3$/ kmol / min and $C_{A0}$ = 1 kmol/m$^3$.

Hence:

$$-r_A = kC_A^2$$

Since this is a liquid phase reaction, $C_A = C_{A0}(1-X) $

Hence,

$$-r_A = k C_{A0}^2(1-X)^2 $$

In [ ]:
k = 0.1
CA0 = 1

## Maximum mixedness model

For the maximum mixedness model,

$$\frac{dX}{dz} = \frac{-r_A}{C_{A0}} - \frac{E(\bar{T}-z)}{1-F(\bar{T}-z)}X $$

where $z = t$ and $\bar{T}$ = 14 min. To do this, we need to be able to evaluate $E(t)$ and $F(t)$ at any value of $t$. To do this, we could use interpolation or regression.

Interpolation is usually the better choice because it uses the data directly.

Normally we would use Lagrange polynomials, but they are "too big" for this problem. So, let's use a lower level of interpolation. (I find that linear and quadratic give the same result, so I'm going to use the simpler one.)

In [ ]:
# bounds_error = False prevents an error message if the solver goes out of bounds.
# fill_value = 0 means: if the solver goes out of bounds, the function value = 0.
# fill_value = (0,1) means: if the solver goes "to the left" of the initial value, set the function to 0, and if it goes "to the right" of the final value, set the function to 1.
E_func = interp1d(t, E, kind='linear', bounds_error=False, fill_value=0)
F_func = interp1d(t, F, kind='linear', bounds_error=False, fill_value=(0.0,1.0))

Now, we can write a function to solve the maxium mixedness model (feel free to adapt it "free of charge," i.e., without losing innovativeness points, in your in-class activity and practice problem solutions):

In [ ]:
def max_mixedness_ode(z, X):
  """
  z: independent variable (integrating backwards from 14 to 0)
  X: conversion array (solve_ivp expects a list/array)
  """
  time_arg = 14.0 - z  # Represents (\bar{T} - z)

  # Look up E and F values at the current time argument
  E_val = E_func(time_arg)
  F_val = F_func(time_arg)

  # rate term: -rA/CA0
  rate = k * CA0**2 * (1.0 - X[0])**2
  rate_term = rate / CA0

  # SAFETY: Prevent division by zero as F(t) approaches 1.0 at the end
  denom = 1.0 - F_val
  if denom < 1e-6:
      denom = 1e-6

  dXdz = rate_term - (E_val / denom) * X[0]
  return [dXdz]

z_span = (0.0, 14.0)
X_initial = [0.0]

sol = solve_ivp(max_mixedness_ode, z_span, X_initial, method='Radau')

X_mm = sol.y[0][-1]

print(f"X = {X_mm:.2f}")

X = 0.32
